# 04 | Panel Granger Causality: First-Differenced Analysis

This notebook runs panel Granger causality tests for each of the 9 technologies.
Both series are first-differenced within country to remove trends, and country
fixed effects soak up time-invariant heterogeneity. We test both directions:

- **Direction A:** does past Δspending help predict Δpubs?
- **Direction B:** does past Δpubs help predict Δspending?


## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

panel = pd.read_csv("../data/processed/merged_panel.csv").sort_values(["country", "technology", "year"])
TECHS = sorted(panel["technology"].unique())


## First-difference within country-technology

In [ ]:
def diff_within(df):
    df = df.sort_values("year").copy()
    df["d_spending"] = df["spending_usd_ppp_millions"].diff()
    df["d_pubs"]     = df["pub_count"].diff()
    return df

panel = panel.groupby(["country", "technology"], group_keys=False).apply(diff_within)


## CCF on differenced series — 3×3 grid

In [ ]:
def ccf(x, y, max_lag=8):
    x = np.asarray(x); y = np.asarray(y)
    x = (x - x.mean()) / (x.std() + 1e-12); y = (y - y.mean()) / (y.std() + 1e-12)
    n = len(x)
    return [np.sum(x[:n-k] * y[k:]) / n for k in range(-max_lag, max_lag + 1)]

LAGS = list(range(-8, 9))
fig, axes = plt.subplots(3, 3, figsize=(13, 10), sharex=True)
for ax, tech in zip(axes.flat, TECHS):
    sub = (
        panel.query("technology == @tech")
             .dropna(subset=["d_spending", "d_pubs"])
             .groupby("year", as_index=False)
             .agg(d_spending=("d_spending", "mean"), d_pubs=("d_pubs", "mean"))
    )
    cc = ccf(sub["d_spending"], sub["d_pubs"])
    ax.bar(LAGS, cc, color="steelblue")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(tech, fontsize=10)
fig.suptitle("Differenced CCF (negative lag = spending leads pubs)", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/ccf_differenced_grid.png", dpi=150, bbox_inches="tight")
plt.show()


## Panel Granger regression with country fixed effects

For each (technology, direction):

```
Δy_{c,t} = α_c + Σ β_k Δx_{c,t-k} + Σ γ_k Δy_{c,t-k} + ε
```

with K=4 lags. F-test on the joint significance of β coefficients.


In [ ]:
def build_lags(df, col, lags):
    out = pd.DataFrame(index=df.index)
    for k in range(1, lags + 1):
        out[f"{col}_l{k}"] = df.groupby(["country", "technology"])[col].shift(k)
    return out


def panel_granger(df, y_col, x_col, lags=4):
    Y = df[y_col]
    Xy = build_lags(df, y_col, lags)
    Xx = build_lags(df, x_col, lags)
    country_dummies = pd.get_dummies(df["country"], drop_first=True).astype(float)
    X_full = pd.concat([Xy, Xx, country_dummies], axis=1)
    X_rest = pd.concat([Xy, country_dummies], axis=1)

    mask = pd.concat([Y, X_full], axis=1).dropna().index
    Y, X_full, X_rest = Y.loc[mask], X_full.loc[mask], X_rest.loc[mask]

    full = sm.OLS(Y, sm.add_constant(X_full)).fit()
    rest = sm.OLS(Y, sm.add_constant(X_rest)).fit()

    rss_f = (full.resid ** 2).sum()
    rss_r = (rest.resid ** 2).sum()
    df_n = lags
    df_d = full.df_resid
    F = ((rss_r - rss_f) / df_n) / (rss_f / df_d)
    p = 1 - sm.stats.stattools.stats.f.cdf(F, df_n, df_d) if False else None
    from scipy.stats import f as f_dist
    p = 1 - f_dist.cdf(F, df_n, df_d)
    return F, p, full.df_resid, full.rsquared


results = []
for tech in TECHS:
    sub = panel.query("technology == @tech").copy()
    if len(sub) < 200:
        continue
    F_a, p_a, dof_a, r2_a = panel_granger(sub, "d_pubs", "d_spending")
    F_b, p_b, dof_b, r2_b = panel_granger(sub, "d_spending", "d_pubs")
    results.append({
        "technology": tech,
        "F_spend_to_pubs": F_a, "p_spend_to_pubs": p_a,
        "F_pubs_to_spend": F_b, "p_pubs_to_spend": p_b,
        "n_obs": dof_a, "r2_full_A": r2_a,
    })

res = pd.DataFrame(results).sort_values("F_spend_to_pubs", ascending=False)
res.to_csv("../results/granger_results.csv", index=False)
res


## Typology classification

In [ ]:
def classify(row, alpha=0.05):
    a = row["p_spend_to_pubs"] < alpha
    b = row["p_pubs_to_spend"] < alpha
    if a and b:   return "Bidirectional"
    if a and not b: return "Spending → Publications"
    if b and not a: return "Publications → Spending"
    return "No robust link"

res["typology"] = res.apply(classify, axis=1)
res[["technology", "typology", "F_spend_to_pubs", "p_spend_to_pubs", "F_pubs_to_spend", "p_pubs_to_spend"]]


## Summary figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
order = res.sort_values("F_spend_to_pubs")["technology"]

axes[0].barh(order, res.set_index("technology").loc[order, "F_spend_to_pubs"], color="steelblue")
axes[0].set_xlabel("F-stat: spending → publications")
axes[0].set_title("Direction A")

axes[1].barh(order, res.set_index("technology").loc[order, "F_pubs_to_spend"], color="firebrick")
axes[1].set_xlabel("F-stat: publications → spending")
axes[1].set_title("Direction B")

for ax in axes:
    ax.axvline(2.5, color="grey", ls="--", lw=0.8, label="approx p=0.05")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../figures/summary_figure.png", dpi=150, bbox_inches="tight")
plt.show()


## Output

- `../results/granger_results.csv`, F-stats, p-values, and typology per technology.
- `../figures/ccf_differenced_grid.png`, `summary_figure.png`.

**Findings:** Hydrogen is bidirectional. Wind, Ocean, CO2 capture, Nuclear show
spending→publications. Solar shows the reverse (publications→spending).
Mature technologies (Hydropower) show no robust link. These results are
stress-tested in notebook 05.
